# Tone Generation in an Unsupported Language — experiment notebook

Companion code for the paper *Tone Generation in an Unsupported Language: Evaluating English-to-Limbum Translation with NLLB-200* (AfriLang AI 2026).
Running this notebook end-to-end produces every number, table and figure in the paper
(`tables/*.tex`, `results/numbers.tex`, `figs/*.pdf`). Copy the `paper_assets/` folder
next to the `.tex` file and recompile: the `\res{...}` placeholders fill themselves in.

**What it runs**

| Step | What | Paper section |
|---|---|---|
| 0 | Load corpus, leakage-safe split | §3.1 |
| 1 | Orthographic + tokenizer audit (no training) | §3.2, RQ1 |
| 2 | NLLB-200 En→Limbum: **Direct**, **Explicit tone tokens**, **Stripped** | §4, RQ2–3 |
| 3 | Restorers: **lexical**, **ByT5**, **source-aware ByT5** (+ oracle eval) | §4.4, RQ3 |
| 4 | Metrics, paired bootstrap, error analysis | §5, §7, RQ4 |
| 5 | Export LaTeX tables / macros / figure / human-eval sheet | — |

**Runtime.** Colab T4 (Runtime → Change runtime type → T4 GPU). Approximate time per NLLB
run 30–50 min, per ByT5 restorer 45–75 min. Every run saves its outputs and is **skipped
on re-run**, so if Colab disconnects just run all cells again. Set `QUICK=True` first to
smoke-test the whole pipeline in ~15 minutes on a tiny subset.

**Data format.** A CSV/TSV with one row per pair:
`lmp` (Limbum, tone-marked), `eng` (English), `source` (`nt` or `dict`), and optionally
`ref` (verse id such as `MAT 5:3`; enables a book-level split). Rename columns in the config
cell if yours differ.

In [1]:
# If Colab reports a dependency conflict and asks to restart, restart the runtime
# and continue from the next cell (the install persists for the session).
!pip -q install "transformers==4.46.3" "sentencepiece>=0.2.0" "sacrebleu==2.4.3" "protobuf"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 805.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 15.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import transformers; print(transformers.__version__)

4.46.3


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd

nt = pd.read_csv("/content/drive/MyDrive/Limbum Research/nt_cleaned.csv")    # adjust file names
dic = pd.read_csv("/content/drive/MyDrive/Limbum Research/limbum_dictionary_cleaned.csv")

# rename to the notebook's expected names — change the left-hand names to match your files
nt = nt.rename(columns={"Limbum": "lmp", "English": "eng"})            # add "Reference": "ref" if you have it
dic = dic.rename(columns={"Limbum": "lmp", "English": "eng"})

dic = dic.drop(columns=[c for c in dic.columns if "len" in c.lower()])  # drop the length columns

nt["source"] = "nt"
dic["source"] = "dict"

keep = ["lmp", "eng", "source"] + (["ref"] if "ref" in nt.columns else [])
merged = pd.concat([nt, dic], ignore_index=True).reindex(columns=keep)
merged = merged.dropna(subset=["lmp", "eng"])
merged.to_csv("/content/drive/MyDrive/Limbum Research/limbum_english.csv", index=False, encoding="utf-8")

print(merged.source.value_counts())   # expect roughly 7380 nt / 1457 dict

source
nt      7386
dict    1456
Name: count, dtype: int64


In [ ]:
merged.head()

,lmp,eng,source
0,Ñwaè' ca yi ce suñ aèmboè raèraâ Jisos Kraâis ...,"The book of the genealogy of Jesus Christ, the...",nt
1,"Abraèham aè m bi Aisïìk, Aisïìk aè bi Jakoèp, ...",Abraham became the father of Isaac. Isaac beca...,nt
2,Judaè aè m bi Páráès ba Saráè ce yaèp ma aè m ...,Judah became the father of Perez and Zerah by ...,nt
3,"Ram aè m bi AÈminadaèp, AÈminadaèp aè bi Nasho...",Ram became the father of Amminadab. Amminadab ...,nt
4,Sarmoèn aè m bi Buwaès ce yïì ma aè m ba Rahaè...,Salmon became the father of Boaz by Rahab. Boa...,nt


In [ ]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8842 entries, 0 to 8841
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   lmp     8842 non-null   object
 1   eng     8842 non-null   object
 2   source  8842 non-null   object
dtypes: object(3)
memory usage: 207.4+ KB


## Configuration

In [4]:
CFG = dict(
    # ---- data ----
    DATA_PATH="/content/drive/MyDrive/Limbum Research/limbum_english.csv",  # upload via the Files pane, or point into Drive
    COL_LMP="lmp", COL_ENG="eng", COL_SOURCE="source", COL_REF="ref",
    # ---- persistence (strongly recommended: Colab sessions die) ----
    USE_DRIVE=True,
    OUT_DIR_DRIVE="/content/drive/MyDrive/Limbum Research/limbum_tone_unicode",
    OUT_DIR_LOCAL="/content/limbum_tone",
    # ---- models ----
    NLLB_MODEL="facebook/nllb-200-distilled-600M",
    BYT5_MODEL="google/byt5-small",
    SRC_LANG="eng_Latn",
    NEW_LANG="lmp_Latn",
    PROXY_LANGS=["swh_Latn", "lug_Latn", "lin_Latn"],  # new-tag init = mean of these (held fixed)
    # ---- orthography: combining marks treated as TONE (check the audit table!) ----
    # grave, acute, circumflex, macron, caron. Non-tone marks are never stripped.
    TONE_MARKS=["\u0300", "\u0302", "\u030C"],   # low (grave), falling (circumflex), rising (caron)
    # ---- experiment ----
    NLLB_SEEDS=[1, 2, 3],
    RESTORER_SEEDS=[1],
    SYSTEMS=["direct", "explicit", "stripped"],
    QUICK=False,  # True = tiny smoke test (300 train pairs, 1 epoch, 60 test sentences)
    # ---- NLLB hyper-parameters ----
    NLLB_LR=1e-4, NLLB_BS=8, NLLB_ACCUM=2, NLLB_EPOCHS=8, NLLB_PATIENCE=2, NLLB_WARMUP=200,
    MAX_SRC_TOK=128, MAX_TGT_TOK=256, BEAMS=4, GEN_BS=16, DEV_EVAL_N=300,
    FP16=True,
    # ---- ByT5 restorer hyper-parameters ----
    BYT5_LR=5e-4, BYT5_BS=4, BYT5_ACCUM=4, BYT5_EPOCHS=6, BYT5_PATIENCE=2, BYT5_WARMUP=200,
    CHUNK_WORDS=30, ENG_MAX_CHARS=250, BYT5_MAX_IN=640, BYT5_MAX_OUT=448, RESTORE_BS=24,
    # ---- evaluation ----
    BOOTSTRAP_N=1000, HUMAN_EVAL_N=100,
)

In [5]:
import os
if CFG["USE_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = CFG["OUT_DIR_DRIVE"]
else:
    OUT = CFG["OUT_DIR_LOCAL"]
for sub in ["runs", "restorers", "audit", "paper_assets/tables", "paper_assets/results",
            "paper_assets/figs", "human_eval"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
print("Outputs ->", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs -> /content/drive/MyDrive/Limbum Research/limbum_tone_unicode


## Utilities: normalization, tone stripping, word handling

In [6]:
import re, json, math, random, gc, time, unicodedata as ud, difflib
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

TONE = set(CFG["TONE_MARKS"])
PUNCT = '.,;:!?"“”‘«»()[]{}…—–-'      # apostrophes ' and ’ are LETTERS in Limbum orthography
SENT_END = set(".!?;:")
_punct_re = re.escape(PUNCT)
AFFIX_RE = re.compile(rf"^([{_punct_re}]*)(.*?)([{_punct_re}]*)$", re.S)

def nfc(s): return ud.normalize("NFC", s)
def nfd(s): return ud.normalize("NFD", s)

def strip_tones(s):
    """Remove only configured tone marks; keep base letters and non-tone diacritics."""
    return nfc("".join(ch for ch in nfd(s) if ch not in TONE))

def count_marks(s, only_tone=True):
    d = nfd(s)
    if only_tone:
        return sum(1 for ch in d if ch in TONE)
    return sum(1 for ch in d if ud.category(ch) == "Mn")

def clean_ws(s): return re.sub(r"\s+", " ", s).strip()

def split_affix(token):
    m = AFFIX_RE.match(token)
    return m.group(1), m.group(2), m.group(3)

def word_items(s):
    """[(core_word, is_phrase_final)] — punctuation stripped from word edges only."""
    toks = s.split()
    out = []
    for i, t in enumerate(toks):
        pre, core, post = split_affix(t)
        if core:
            out.append([core, any(c in SENT_END for c in post)])
        elif out and any(c in SENT_END for c in t):   # punctuation separated by a space
            out[-1][1] = True
    if out:
        out[-1][1] = True
    return [(w, f) for w, f in out]

def key(w):  # lexicon key: tone-stripped, lower-cased
    return strip_tones(w).lower()

def set_seed(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass

def save_lines(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(l.replace("\n", " ") for l in lines) + "\n")

def load_lines(path):
    with open(path, encoding="utf-8") as f:
        return [l.rstrip("\n") for l in f]

In [ ]:
import re, os, unicodedata as ud, pandas as pd
p = os.path.join(OUT, "split.csv")
df = pd.read_csv(p, dtype=str)
TM = "\u0300\u0302\u030C"
def fix_marks(s):
    d = ud.normalize("NFD", s)
    d = re.sub(rf"([mnŋMNŊ]) ([{TM}])", r"\1\2", d)     # "M ̀" -> "M̀" (tone on syllabic nasal)
    d = re.sub(rf"([{TM}])\1+", r"\1", d)               # duplicated marks, e.g. "nà̀" -> "nà"
    d = re.sub(rf"(^|\s)[{TM}]+", r"\1", d)             # any other stranded marks
    return ud.normalize("NFC", re.sub(r"\s+", " ", d).strip())
before = df.lmp.copy()
df["lmp"] = df.lmp.map(fix_marks)
changed = before != df.lmp
print(changed.sum(), "rows fixed", df[changed].source.value_counts().to_dict())
for a, b in list(zip(before[changed], df.lmp[changed]))[:5]: print(a, "\n→", b, "\n")
df.to_csv(p, index=False)

43 rows fixed {'dict': 29, 'nt': 14}
Josɛ̀f à ka' lɛ' efa mmirlò̀, a gee yuu ce ncìndap Tata Nyʉ̀ à m suŋ ye à rlo nà, a lor ŋgwa yì a du a ye fa yê là'. 
→ Josɛ̀f à ka' lɛ' efa mmirlò, a gee yuu ce ncìndap Tata Nyʉ̀ à m suŋ ye à rlo nà, a lor ŋgwa yì a du a ye fa yê là'. 

Josɛ̀f à lɛ' efa mmirlò̀, a lor muu ba ma muu nètʉ', a du à to' Ijìp, 
→ Josɛ̀f à lɛ' efa mmirlò, a lor muu ba ma muu nètʉ', a du à to' Ijìp, 

E beesì enɛ̂, "A saŋ mbe Ŋwâ' Nyʉ̀ enɛ̂, "Ŋwɛ̀ e ke cu ntèe ba njo' brɛ̂d cà'cà' ka'. E ke cu ba njo' blàa ce vi ke fʉʉ ekù̀̀̀ cûu Nyʉ̀." " 
→ E beesì enɛ̂, "A saŋ mbe Ŋwâ' Nyʉ̀ enɛ̂, "Ŋwɛ̀ e ke cu ntèe ba njo' brɛ̂d cà'cà' ka'. E ke cu ba njo' blàa ce vi ke fʉʉ ekù cûu Nyʉ̀." " 

Yì nsuŋ à du à bbu' mla' wɛɛwɛ à ŋgôŋ Sirìyà, bèe a vʉ nè bèe wɛɛwɛ ce a m ce yaŋ byaŋ jɛɛ jɛ̀ɛ jɛɛ, ba bèe ce mnyor vi m ce yaŋ wowèe, ba bèe ce byʉ' mnyʉ̂mkà' vi m ba njep wowèe, ba bèe ce a m ce gwè kà', ker ba bèe ce mŋgʉ̀' nyor mo' vi m kweŋger. Jisòs à gee wowèe a ke'sè̀. 
→ Yì nsuŋ à du à bb

## Step 0 — Load corpus and split without leakage
NT verses are split **by book** when a `ref` column exists (otherwise by contiguous blocks of
verses), so test verses never sit next to training verses. Dictionary pairs are split randomly.
The split is fixed (seed 0) and saved, so every system sees identical data.

In [7]:
def load_corpus():
    p = CFG["DATA_PATH"]
    df = pd.read_csv(p, sep="\t" if p.endswith((".tsv", ".txt")) else ",", dtype=str)
    ren = {CFG["COL_LMP"]: "lmp", CFG["COL_ENG"]: "eng"}
    if CFG["COL_SOURCE"] in df.columns: ren[CFG["COL_SOURCE"]] = "source"
    if CFG["COL_REF"] in df.columns: ren[CFG["COL_REF"]] = "ref"
    df = df.rename(columns=ren)
    if "source" not in df.columns: df["source"] = "nt"
    df = df.dropna(subset=["lmp", "eng"]).copy()
    df["lmp"] = df["lmp"].map(lambda s: clean_ws(nfc(s)))
    df["eng"] = df["eng"].map(lambda s: clean_ws(nfc(s)))
    df["source"] = df["source"].str.lower().str.strip().map(lambda s: "dict" if s.startswith("dic") else "nt")
    df = df[(df.lmp.str.len() > 0) & (df.eng.str.len() > 0)]
    df = df.drop_duplicates(subset=["lmp", "eng"]).reset_index(drop=True)
    return df

def make_split(df):
    rng = random.Random(0)
    split = pd.Series("train", index=df.index)
    nt = df[df.source == "nt"]
    if "ref" in df.columns and nt["ref"].notna().all():
        def book_of(r):
            m = re.match(r"\s*(\d?\s*[^\d\s:.]+)", r)
            return (m.group(1) if m else r.split()[0]).strip().upper()
        book = nt["ref"].map(book_of)
        books = sorted(book.unique()); rng.shuffle(books)
        sizes = book.value_counts(); target = 0.10 * len(nt)
        test, dev, acc = set(), set(), 0
        for b in books:
            if acc < target: test.add(b); acc += sizes[b]
            elif acc < 2 * target: dev.add(b); acc += sizes[b]
        split[nt.index[book.isin(test).values]] = "test"
        split[nt.index[book.isin(dev).values]] = "dev"
        how = f"book-level (test books: {sorted(test)}; dev books: {sorted(dev)})"
    else:
        blocks = np.array_split(np.array(nt.index), 20)
        order = list(range(20)); rng.shuffle(order)
        for j, bi in enumerate(order):
            if j < 2: split[blocks[bi]] = "test"
            elif j < 4: split[blocks[bi]] = "dev"
        how = "20 contiguous verse blocks (2 test / 2 dev / 16 train)"
    di = list(df[df.source == "dict"].index); rng.shuffle(di)
    n = len(di)
    split[di[: int(0.1 * n)]] = "test"
    split[di[int(0.1 * n): int(0.2 * n)]] = "dev"
    df = df.assign(split=split.values)
    return df, how

split_path = os.path.join(OUT, "split.csv")
if os.path.exists(split_path):
    data = pd.read_csv(split_path, dtype=str); split_how = open(os.path.join(OUT, "split_how.txt")).read()
else:
    data, split_how = make_split(load_corpus())
    data.to_csv(split_path, index=False); open(os.path.join(OUT, "split_how.txt"), "w").write(split_how)

train = data[data.split == "train"].reset_index(drop=True)
dev = data[data.split == "dev"].reset_index(drop=True)
test = data[data.split == "test"].reset_index(drop=True)
if CFG["QUICK"]:
    train = train.sample(min(300, len(train)), random_state=0).reset_index(drop=True)
    dev = dev.sample(min(60, len(dev)), random_state=0).reset_index(drop=True)
    test = test.sample(min(60, len(test)), random_state=0).reset_index(drop=True)
    CFG.update(NLLB_EPOCHS=1, BYT5_EPOCHS=1, BOOTSTRAP_N=100, DEV_EVAL_N=40, HUMAN_EVAL_N=10)
    OUT_RUNS = os.path.join(OUT, "quick")
else:
    OUT_RUNS = OUT
os.makedirs(os.path.join(OUT_RUNS, "runs"), exist_ok=True)
os.makedirs(os.path.join(OUT_RUNS, "restorers"), exist_ok=True)
print("Split:", split_how)
print(data.groupby(["split", "source"]).size().unstack(fill_value=0))

Split: 20 contiguous verse blocks (2 test / 2 dev / 16 train)
source  dict    nt
split             
dev      146   738
test     145   738
train   1165  5897


In [ ]:
import collections
nt_txt = " ".join(data[data.source == "nt"].lmp); di_txt = " ".join(data[data.source == "dict"].lmp)
for name, t in [("NT", nt_txt), ("Dict", di_txt)]:
    print(name, {c: t.count(c) for c in "áÁúñÑèìêâïɛʉŋ"})
print("char before 'è':", collections.Counter(nt_txt[i-1] for i in range(1, len(nt_txt)) if nt_txt[i] == "è").most_common(8))
for s in data.lmp:                       # Luke 11:2–4 (Lord's Prayer), to compare with the Omniglot Unicode text
    if "Taèr" in s or "Tàr" in s: print(s, "\n")

NT {'á': 33114, 'Á': 137, 'ú': 12137, 'ñ': 18842, 'Ñ': 1379, 'è': 84487, 'ì': 7401, 'ê': 4110, 'â': 4894, 'ï': 7902, 'ɛ': 0, 'ʉ': 0, 'ŋ': 0}
Dict {'á': 0, 'Á': 0, 'ú': 0, 'ñ': 0, 'Ñ': 0, 'è': 745, 'ì': 315, 'ê': 164, 'â': 77, 'ï': 0, 'ɛ': 1479, 'ʉ': 564, 'ŋ': 869}
char before 'è': [('a', 27991), ('e', 27099), ('o', 10539), ('á', 9414), ('ú', 5774), ('u', 3645), ('è', 15), ('ù', 3)]
Baná weèe loo neè Taèr ce e túr ñkúè ená e tuusi ñgaâa mfaè' tá a du kep yïì buu efa mñkúè.” 

AÈ m ba aè nfeè' anaè Jisoès aè laa enáê, “Taèr, Tata ce aè túr rbuu ba nshe, m ce yoèñshi wáè njobeè' wáè súèti buu ca ese ñgaâa bfee ba beèe ce a rïìñ buu, a duñshi ba neè boo bká'. 

ÁÈno, Taèr, ca yu ba yuu ce wáè m ce yu' rboèñ aè rgeèe. 

“Yaè Ta aè fa buu wááwá mba máâ mmboè. Ñwáè moè' rïìñ muu caa efa Taèr ka', ñwáè moè' aè rïìñ Taèr caa efa muu ker ba beèe ce muu aè koèñ aè rduèñshi ye neè woweèe ka'. 

Yi yu ená Muu Ñwáè yu Taèr ce e túr mmbomntaa Nyúè.” 

Weèe fa too ñwáè moè' enjep ñgoèñ na ená Taèr ka'

In [ ]:
# %%
# Step 0b — Convert the legacy-font New Testament text to proper Unicode.
# Run AFTER the config cell, the Drive/OUT cell and the utilities cell (no GPU needed).
# It reads the existing split (same train/dev/test rows as your first run), converts the
# Limbum side of NT rows, normalises curly quotes everywhere, and writes a NEW output
# folder, so the legacy runs stay untouched for the legacy-vs-Unicode comparison.
import os, unicodedata as ud
import pandas as pd
from collections import Counter

OLD_OUT = OUT                                                  # legacy results folder
NEW_OUT = OUT.rstrip("/") + "_unicode"                         # new results folder
os.makedirs(NEW_OUT, exist_ok=True)

GRAVE, CIRC = "\u0300", "\u0302"
# legacy tone glyphs, typed AFTER the vowel they belong to (wide and narrow variants)
TONE_GLYPH = {"è": GRAVE, "È": GRAVE, "ì": GRAVE, "Ì": GRAVE,
              "ê": CIRC, "Ê": CIRC, "â": CIRC, "Â": CIRC, "î": CIRC, "Î": CIRC, "û": CIRC, "Û": CIRC}
# legacy letters (verified on Luke 11:2 against the Unicode text on Omniglot)
LETTER = {"á": "ɛ", "Á": "Ɛ", "ú": "ʉ", "Ú": "Ʉ", "ñ": "ŋ", "Ñ": "Ŋ", "ï": "i", "Ï": "I"}
VOWELS = set("aeiouɛʉAEIOUƐɄ")
QUOTES = {"“": '"', "”": '"', "‘": '"', "’": '"', "ʼ": "'"}   # curly quotes are <unk> for NLLB; ʼ = glottal '

conv_stats = Counter(); unattached = []

def legacy_to_unicode(s):
    out = []
    for ch in s:
        if ch in TONE_GLYPH:
            j = len(out) - 1
            while j >= 0 and ud.category(out[j]) == "Mn":     # skip marks already on the vowel
                j -= 1
            if j >= 0 and out[j] in VOWELS:
                out.append(TONE_GLYPH[ch]); conv_stats["tone_attached"] += 1
                continue
            conv_stats["tone_unattached"] += 1; unattached.append(s)
            out.append(ch)                                     # left as-is, listed below for review
        else:
            if ch in LETTER: conv_stats[f"{ch}->{LETTER[ch]}"] += 1
            out.append(LETTER.get(ch, ch))
    return ud.normalize("NFC", "".join(out))

def norm_quotes(s):
    return "".join(QUOTES.get(c, c) for c in s)

old = pd.read_csv(os.path.join(OLD_OUT, "split.csv"), dtype=str)
new = old.copy()
is_nt = new.source == "nt"
new.loc[is_nt, "lmp"] = new.loc[is_nt, "lmp"].map(legacy_to_unicode)
new["lmp"] = new["lmp"].map(lambda s: ud.normalize("NFC", norm_quotes(s)))
new["eng"] = new["eng"].map(norm_quotes)
new.to_csv(os.path.join(NEW_OUT, "split.csv"), index=False)
with open(os.path.join(OLD_OUT, "split_how.txt")) as f_in, open(os.path.join(NEW_OUT, "split_how.txt"), "w") as f_out:
    f_out.write(f_in.read())

# ---------------- checks ----------------
print("Conversion counts:", dict(conv_stats))
print(f"Unattached tone glyphs: {conv_stats['tone_unattached']} (should be ~0)")
for s in list(dict.fromkeys(unattached))[:5]: print("  REVIEW:", s)

print("\nLuke 11:2 check (expected: E m làa nè wowèe enɛ̂, \"Wèe à ka' ce gèe rloo, ...):")
for s in new.lmp:
    if s.startswith("E m làa nè wowèe enɛ̂, \"Wèe"): print(" ", s); break

# (è, ê, ì ... are legitimate Unicode letters after conversion, e.g. e + low tone, so only the
#  legacy-only letters are checked here; stray tone glyphs are caught by the unattached count above)
legacy_left = {c: int(new[is_nt].lmp.str.count(c).sum()) for c in "áÁúÚñÑïÏ"}
print("\nLegacy letters left in NT (all should be 0):", legacy_left)

def marks_by_source(df):
    rows = []
    for src in ["nt", "dict"]:
        t = ud.normalize("NFD", " ".join(df[df.source == src].lmp))
        rows.append(dict(source=src, grave=t.count("\u0300"), acute=t.count("\u0301"),
                         circumflex=t.count("\u0302"), caron=t.count("\u030C"), macron=t.count("\u0304"),
                         tilde=t.count("\u0303"), diaeresis=t.count("\u0308"),
                         eps=t.count("ɛ"), ubar=t.count("ʉ"), eng=t.count("ŋ")))
    return pd.DataFrame(rows)
print("\nCombining marks and special letters per source AFTER conversion:")
display(marks_by_source(new))
print(f"\nWrote {len(new)} rows to {NEW_OUT}/split.csv")

Conversion counts: {'Ñ->Ŋ': 1379, 'tone_attached': 102442, 'ñ->ŋ': 18842, 'á->ɛ': 33114, 'ï->i': 7902, 'ú->ʉ': 12137, 'Á->Ɛ': 137, 'tone_unattached': 13}
Unattached tone glyphs: 13 (should be ~0)
  REVIEW: Kùù Nyùè yi yu foñ aèmboè fwàé rntaa ce e m ce roko sibañ.
  REVIEW: Woweèe aè m laèa enáâ, “E m jaè’ beèe mo’, a moo aè rjaè’ yi tu laè’r. E yu ñkuù beèe Isràèl; e suu laè embe klìntaa seè’ni na, tá seèe bo byemi neè ye.
  REVIEW: E làé ntee njep Nyùk, baná Nyùk ka' koñ ye, e fùr laè ye efa fígà’, njobee’ e m laèa ená è ye Muu Nyùk.
  REVIEW: Yù’ Tata yi yu weèr ye, njobee’ Nyùè aè ce màá ená m fe’shi nsuñ mboèfmbof nneè figaâa fuèp. E tuu màá ená m du suñ beèe ce a yu mbe shaf ená nfeè’ aè gháá ce a be fùr woweèe efa shaf, tá beèe ce a feèp a yá buu, tá beèe ce beèe a ce jar neè woweèe a luñ efa ñga’,
  REVIEW: ker suñ aèmboè nfeè’ ce Tata Nyùè be ghoèni yii beèe.

Luke 11:2 check (expected: E m làa nè wowèe enɛ̂, "Wèe à ka' ce gèe rloo, ...):
  E m làa nè wowèe enɛ̂, "Wèe à ka' du

,source,grave,acute,circumflex,caron,macron,tilde,diaeresis,eps,ubar,eng
0,nt,92980,16,9513,0,0,0,0,33114,12137,18842
1,dict,4197,0,311,118,0,0,0,1479,564,869



Wrote 8842 rows to /content/drive/MyDrive/Limbum Research/limbum_tone_unicode/split.csv


In [ ]:
import os, pandas as pd
OLD_P = os.path.join(OUT, "split.csv")                              # legacy text (OUT = old folder still)
NEW_P = OUT.rstrip("/") + "_unicode/split.csv"
old = pd.read_csv(OLD_P, dtype=str); new = pd.read_csv(NEW_P, dtype=str)
assert len(old) == len(new)
SUSPECT = set("àòùéíóÀÒÙÉÍÓ")                                       # never produced by the legacy font
bad = (old.source == "nt") & old.lmp.map(lambda s: any(c in SUSPECT for c in s))
print(f"{bad.sum()} mixed-encoding NT verses ({100*bad.mean():.1f}% of all rows)")
print("by split:", new[bad].split.value_counts().to_dict())
display(old[bad].lmp.head(8))
new[bad].to_csv(NEW_P.replace("split.csv", "excluded_mixed_encoding.csv"), index=False)
new[~bad].to_csv(NEW_P, index=False)
print("kept", (~bad).sum(), "rows")

13 mixed-encoding NT verses (0.1% of all rows)
by split: {'train': 13}


,lmp
335,Kùù Nyùè yi yu foñ aèmboè fwàé rntaa ce e m ce...
464,"Máè ka' vùé neè ye, fa neè figaâa yá’ni yoè bu..."
811,"Woweèe aè m laèa enáâ, “E m jaè’ beèe mo’, a m..."
812,"E làé ntee njep Nyùk, baná Nyùk ka' koñ ye, e ..."
1683,"Yù’ Tata yi yu weèr ye, njobee’ Nyùè aè ce màá..."
1684,ker suñ aèmboè nfeè’ ce Tata Nyùè be ghoèni yi...
2115,Ñwaè rfaé’ aè kùtí a suñ buu ce woweèe a m laè...
2116,"Ñwaè rfaé’ anaè sè sùtí, a laè enáâ, ‘Mmaâ, má..."


kept 8829 rows


## Step 1a — Orthographic audit (RQ1)
How consistently is tone written? Which marks occur? Is mid tone written with a macron
(linguistic transcription) or left unmarked (practical orthography)? Which tone-stripped word
forms have several toned spellings (real ambiguity *or* inconsistent marking — the CSV is for a
native speaker to adjudicate)?

In [8]:
AUDIT = {}
all_lmp = list(data.lmp)
mark_counter, base_counter = Counter(), Counter()
words_total = words_marked = 0
for s in all_lmp:
    for ch in nfd(s):
        if ud.category(ch) == "Mn": mark_counter[ch] += 1
    for ch in s:
        if ord(ch) > 127 and ud.category(ch) != "Mn" and ch not in PUNCT: base_counter[ch] += 1
    for w, _ in word_items(s):
        words_total += 1
        words_marked += count_marks(w) > 0

mark_tbl = pd.DataFrame([(f"U+{ord(m):04X}", ud.name(m, "?"), c, m in TONE)
                         for m, c in mark_counter.most_common()],
                        columns=["codepoint", "name", "count", "treated_as_tone"])
base_tbl = pd.DataFrame([(ch, f"U+{ord(ch):04X}", ud.name(ch, "?"), c) for ch, c in base_counter.most_common(40)],
                        columns=["char", "codepoint", "name", "count"])
mark_tbl.to_csv(os.path.join(OUT, "audit/combining_marks.csv"), index=False)
base_tbl.to_csv(os.path.join(OUT, "audit/non_ascii_letters.csv"), index=False)
display(mark_tbl); display(base_tbl.head(20))

AUDIT["WordsMarkedPct"] = 100 * words_marked / max(1, words_total)
AUDIT["MarksPerWord"] = sum(c for m, c in mark_counter.items() if m in TONE) / max(1, words_total)
AUDIT["MacronCount"] = mark_counter.get("\u0304", 0)
AUDIT["AcuteCount"] = mark_counter.get("\u0301", 0)
AUDIT["GraveCount"] = mark_counter.get("\u0300", 0)
for src in ["nt", "dict"]:
    tw = mk = 0
    for s in data[data.source == src].lmp:
        for w, _ in word_items(s):
            tw += 1; mk += count_marks(w) > 0
    AUDIT[f"WordsMarkedPct{src.upper()}"] = 100 * mk / max(1, tw)
unknown_marks = [f"U+{ord(m):04X}" for m in mark_counter if m not in TONE]
print("Combining marks NOT treated as tone:", unknown_marks or "none",
      "\n→ if any of these encode tone, add them to CFG['TONE_MARKS'] and re-run from the top.")

# Toned variants per stripped type (training data only => no test leakage)
lex = defaultdict(Counter)
for s in train.lmp:
    for w, _ in word_items(s):
        lex[key(w)][nfc(w).lower()] += 1
lex_seen = set(lex)
def contested(c, min_share=0.10, min_count=3):
    if len(c) < 2: return False
    second = c.most_common(2)[1][1]
    return second >= min_count and second / sum(c.values()) >= min_share
lex_amb = {k for k, c in lex.items() if contested(c)}
freq5 = [k for k, c in lex.items() if sum(c.values()) >= 5]
AUDIT["AmbTypesPct"] = 100 * len(lex_amb) / max(1, len(lex))
AUDIT["AmbTypesPctFreqFive"] = 100 * sum(k in lex_amb for k in freq5) / max(1, len(freq5))
tok_total = sum(sum(c.values()) for c in lex.values())
AUDIT["AmbTokensPct"] = 100 * sum(sum(lex[k].values()) for k in lex_amb) / max(1, tok_total)
amb_rows = sorted(((k, sum(c.values()), "; ".join(f"{f}×{n}" for f, n in c.most_common()))
                   for k, c in lex.items() if k in lex_amb), key=lambda r: -r[1])
pd.DataFrame(amb_rows, columns=["stripped_form", "freq", "toned_variants"]).to_csv(
    os.path.join(OUT, "audit/ambiguous_types_for_native_speaker_review.csv"), index=False)
print({k: round(v, 2) if isinstance(v, float) else v for k, v in AUDIT.items()})
print("Top ambiguous types:", amb_rows[:15])

,codepoint,name,count,treated_as_tone
0,U+0300,COMBINING GRAVE ACCENT,96979,True
1,U+0302,COMBINING CIRCUMFLEX ACCENT,9814,True
2,U+030C,COMBINING CARON,118,True


,char,codepoint,name,count
0,ɛ,U+025B,LATIN SMALL LETTER OPEN E,34558
1,à,U+00E0,LATIN SMALL LETTER A WITH GRAVE,29344
2,è,U+00E8,LATIN SMALL LETTER E WITH GRAVE,27795
3,ŋ,U+014B,LATIN SMALL LETTER ENG,19682
4,ʉ,U+0289,LATIN SMALL LETTER U BAR,12701
5,ò,U+00F2,LATIN SMALL LETTER O WITH GRAVE,10987
6,ì,U+00EC,LATIN SMALL LETTER I WITH GRAVE,7712
7,ù,U+00F9,LATIN SMALL LETTER U WITH GRAVE,3984
8,â,U+00E2,LATIN SMALL LETTER A WITH CIRCUMFLEX,3058
9,ê,U+00EA,LATIN SMALL LETTER E WITH CIRCUMFLEX,1941


Combining marks NOT treated as tone: none 
→ if any of these encode tone, add them to CFG['TONE_MARKS'] and re-run from the top.
{'WordsMarkedPct': 40.77, 'MarksPerWord': 0.43, 'MacronCount': 0, 'AcuteCount': 0, 'GraveCount': 96979, 'WordsMarkedPctNT': 40.62, 'WordsMarkedPctDICT': 44.31, 'AmbTypesPct': 9.96, 'AmbTypesPctFreqFive': 24.36, 'AmbTokensPct': 35.39}
Top ambiguous types: [('a', 21092, 'a×11736; à×9351; â×5'), ('enɛ', 4123, 'enɛ×2291; enɛ̂×1832'), ('yi', 3502, 'yi×2511; yì×902; yî×89'), ('laa', 2528, 'laa×1293; làa×1214; lâa×21'), ('gee', 1936, 'gèe×1396; gee×507; gêe×32; gěe×1'), ('na', 1641, 'nà×1017; na×624'), ("mo'", 1613, "mò'×855; mo'×738; mô'×20"), ('jisos', 1581, 'jisòs×1273; jisos×305; jisôs×3'), ('du', 1321, 'du×670; dù×582; dû×69'), ('ker', 1053, 'ker×861; kèr×192'), ('ku', 935, 'ku×766; kù×165; kû×3; kǔ×1'), ('vʉ', 909, 'vʉ̀×514; vʉ×306; vʉ̂×87; vʉ̌×2'), ('ke', 872, 'ke×670; kè×202'), ('riŋ', 841, 'rìŋ×532; riŋ×287; rîŋ×22'), ('se', 650, 'sê×569; se×81')]


## Step 1b — Can NLLB-200's tokenizer even *represent* Limbum tone? (RQ1)
Round trip: text → NLLB SentencePiece ids → text. If tone marks do not survive this, no
model trained with this tokenizer can output them. We also test the explicit-tone-token
encoding used by the **Explicit** system (it must be lossless by construction).

In [ ]:
# %%
# Step 1b (fast version): batched tokenization, cached segment encoding,
# a precomputed id->piece table, and a progress bar on every pass.
import time, functools
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t0 = time.time()
print("Loading NLLB tokenizer ...")
tok = AutoTokenizer.from_pretrained(CFG["NLLB_MODEL"])
VOCAB = tok.get_vocab()
NTOK = len(tok)
ID2PIECE = tok.convert_ids_to_tokens(list(range(NTOK)))   # one-off table: id -> piece string
UNK = tok.unk_token_id
ENG_ID = tok.convert_tokens_to_ids(CFG["SRC_LANG"])
for p in CFG["PROXY_LANGS"] + [CFG["SRC_LANG"]]:
    assert tok.convert_tokens_to_ids(p) != UNK, f"{p} is not an NLLB language code"
MARKS = list(CFG["TONE_MARKS"])
SPECIAL_IDS = set(tok.all_special_ids)
STRANDED = re.compile(rf"(^|\s)[{''.join(MARKS)}]+")
print(f"  tokenizer ready in {time.time() - t0:.1f}s")

def ids_to_text(ids, extra=None, skip=()):
    """Deterministic decoder shared by all systems (no reliance on tokenizer.decode quirks)."""
    extra = extra or {}
    out = []
    for i in ids:
        i = int(i)
        if i in skip or i in SPECIAL_IDS: continue
        if i in extra: out.append(extra[i]); continue
        if i < NTOK: out.append(ID2PIECE[i])
    s = "".join(out).replace("▁", " ")
    s = STRANDED.sub(r"\1", nfd(s))          # drop marks stranded at word starts
    return clean_ws(nfc(s))

def encode_plain(text):
    return tok(text, add_special_tokens=False).input_ids

def encode_plain_batch(texts, bs=512, desc=None):
    out = []
    for i in tqdm(range(0, len(texts), bs), desc=desc, disable=desc is None):
        out.extend(tok(texts[i:i + bs], add_special_tokens=False).input_ids)
    return out

@functools.lru_cache(maxsize=None)
def _piece_ids(seg, at_word_start):
    pieces = tok.tokenize(seg)
    if not at_word_start and pieces:
        p0 = pieces[0]
        if p0 == "▁":
            pieces = pieces[1:]
        elif p0.startswith("▁"):
            cand = p0[1:]
            pieces = ([cand] if cand in VOCAB else [c if c in VOCAB else tok.unk_token for c in cand]) + pieces[1:]
    return tuple(tok.convert_tokens_to_ids(pieces))

def encode_explicit(text, mark_ids):
    """NFD text -> SentencePiece ids for base text + one dedicated id per tone mark."""
    ids, buf, first = [], "", True
    for ch in nfd(text):
        if ch in mark_ids:
            if buf:
                ids.extend(_piece_ids(buf, first or buf[0].isspace())); buf = ""
            ids.append(mark_ids[ch]); first = False
        else:
            buf += ch
    if buf:
        ids.extend(_piece_ids(buf, first or buf[0].isspace()))
    return ids

# hypothetical ids for the round-trip test (same scheme as in training)
_E = NTOK + 10
RT_MARK_IDS = {m: _E + k for k, m in enumerate(MARKS)}
RT_EXTRA = {v: k for k, v in RT_MARK_IDS.items()}

texts = list(train.lmp)
t0 = time.time()
ids_nfc = encode_plain_batch([nfc(s) for s in texts], desc="encode NFC")
ids_nfd = encode_plain_batch([nfd(s) for s in texts], desc="encode NFD")
ids_str = encode_plain_batch([strip_tones(s) for s in texts], desc="encode stripped")
ids_exp = [encode_explicit(s, RT_MARK_IDS) for s in tqdm(texts, desc="encode explicit")]
print(f"  encoding done in {time.time() - t0:.1f}s")

def mark_name(m):
    return ud.name(m, "?").replace("COMBINING ", "").split()[0].lower()

def rt_stats(texts, ids_list, mode, extra=None):
    exact = surv = tot = unk = ntok = nw = 0
    per_mark, per_mark_tot = Counter(), Counter()
    for s, ids in tqdm(zip(texts, ids_list), total=len(texts), desc=f"round trip [{mode}]"):
        back = ids_to_text(ids, extra)
        exact += nfc(back) == nfc(s)
        unk += sum(i == UNK for i in ids); ntok += len(ids); nw += len(s.split())
        o = Counter(ch for ch in nfd(s) if ch in TONE)
        b = Counter(ch for ch in nfd(back) if ch in TONE)
        for m in o:
            per_mark_tot[m] += o[m]; per_mark[m] += min(o[m], b[m])
        tot += sum(o.values()); surv += sum(min(o[m], b[m]) for m in o)
    return dict(mode=mode, exact_roundtrip_pct=100 * exact / len(texts),
                mark_survival_pct=100 * surv / max(1, tot),
                unk_per_100_tokens=100 * unk / max(1, ntok), tokens_per_word=ntok / max(1, nw),
                **{f"survival_{mark_name(m)}": 100 * per_mark[m] / per_mark_tot[m] for m in per_mark_tot})

rt = pd.DataFrame([rt_stats(texts, ids_nfc, "plain_nfc"),
                   rt_stats([nfd(s) for s in texts], ids_nfd, "plain_nfd"),
                   rt_stats(texts, ids_exp, "explicit", RT_EXTRA)])
rt.to_csv(os.path.join(OUT, "audit/tokenizer_roundtrip.csv"), index=False)
display(rt)

AUDIT["RTExactNFC"] = rt.loc[0, "exact_roundtrip_pct"]
AUDIT["RTMarkSurvNFC"] = rt.loc[0, "mark_survival_pct"]
AUDIT["RTMarkSurvNFD"] = rt.loc[1, "mark_survival_pct"]
AUDIT["RTExactExplicit"] = rt.loc[2, "exact_roundtrip_pct"]
AUDIT["UnkPerHundredNFC"] = rt.loc[0, "unk_per_100_tokens"]
AUDIT["FertToned"] = rt.loc[0, "tokens_per_word"]
AUDIT["FertStripped"] = sum(map(len, ids_str)) / max(1, sum(len(s.split()) for s in texts))
AUDIT["FertExplicit"] = rt.loc[2, "tokens_per_word"]
AUDIT["TruncPctExplicit"] = 100 * np.mean([len(x) + 2 > CFG["MAX_TGT_TOK"] for x in ids_exp])

shown = 0
for s, ids in zip(texts, ids_nfc):
    b = ids_to_text(ids)
    if b != s:
        print("ORIG:", s, "\nBACK:", b, "\n"); shown += 1
        if shown == 8: break
json.dump(AUDIT, open(os.path.join(OUT, "audit/audit.json"), "w"), indent=1)
print("Step 1b complete.")

Loading NLLB tokenizer ...
  tokenizer ready in 2.4s


encode NFC:   0%|          | 0/14 [00:00<?, ?it/s]

encode NFD:   0%|          | 0/14 [00:00<?, ?it/s]

encode stripped:   0%|          | 0/14 [00:00<?, ?it/s]

encode explicit:   0%|          | 0/7062 [00:00<?, ?it/s]

  encoding done in 2.5s


round trip [plain_nfc]:   0%|          | 0/7062 [00:00<?, ?it/s]

round trip [plain_nfd]:   0%|          | 0/7062 [00:00<?, ?it/s]

round trip [explicit]:   0%|          | 0/7062 [00:00<?, ?it/s]

,mode,exact_roundtrip_pct,mark_survival_pct,unk_per_100_tokens,tokens_per_word,survival_grave,survival_circumflex,survival_caron
0,plain_nfc,100.0,100.0,0.0,1.861872,100.0,100.0,100.0
1,plain_nfd,100.0,100.0,0.0,1.861872,100.0,100.0,100.0
2,explicit,100.0,100.0,0.0,2.158938,100.0,100.0,100.0


Step 1b complete.


## Step 2 — NLLB-200 fine-tuning: Direct / Explicit / Stripped
All three systems share: same split, same new `lmp_Latn` target tag initialised as the mean
of the proxy-language tag embeddings, same optimiser, schedule, early stopping and decoding.
They differ only in the **target representation**.

In [22]:
from transformers.optimization import Adafactor
from torch.optim.lr_scheduler import LambdaLR
import sacrebleu
from sacrebleu.metrics import CHRF, BLEU
DEV = "cuda" if torch.cuda.is_available() else "cpu"
assert DEV == "cuda", "Switch the runtime to a GPU (T4)."
CHRF_M = CHRF(word_order=2)
BLEU_M = BLEU()

def chrf(h, r): return CHRF_M.corpus_score(h, [r]).score

def setup_nllb(system):
    model = AutoModelForSeq2SeqLM.from_pretrained(CFG["NLLB_MODEL"])
    E = model.get_input_embeddings().weight.shape[0]
    names = [CFG["NEW_LANG"]] + ([f"<tone:{ord(m):04X}>" for m in MARKS] if system == "explicit" else [])
    model.resize_token_embeddings(E + len(names))
    W = model.get_input_embeddings().weight.data
    ids = {n: E + k for k, n in enumerate(names)}
    proxy = torch.stack([W[tok.convert_tokens_to_ids(p)] for p in CFG["PROXY_LANGS"]])
    W[ids[CFG["NEW_LANG"]]] = proxy.mean(0)
    mark_ids = {}
    if system == "explicit":
        for m in MARKS:
            mid = ids[f"<tone:{ord(m):04X}>"]; mark_ids[m] = mid
            if m in VOCAB and VOCAB[m] != UNK:
                W[mid] = W[VOCAB[m]].clone()                 # embedding of the raw combining char
            else:
                cands = [i for p, i in VOCAB.items() if m in nfd(p)][:200]
                W[mid] = W[cands].mean(0) if cands else torch.randn_like(W[0]) * W.std()
    return model, ids[CFG["NEW_LANG"]], mark_ids

def target_of(text, system):
    return strip_tones(text) if system == "stripped" else text

def encode_tgt(text, system, mark_ids):
    return encode_explicit(text, mark_ids) if system == "explicit" else encode_plain(nfc(text))

def build_pairs(df, system, lmp_id, mark_ids):
    pairs = []
    for e, l in zip(df.eng, df.lmp):
        src = [ENG_ID] + encode_plain(e)[: CFG["MAX_SRC_TOK"] - 2] + [tok.eos_token_id]
        tgt = [lmp_id] + encode_tgt(target_of(l, system), system, mark_ids)[: CFG["MAX_TGT_TOK"] - 2] + [tok.eos_token_id]
        pairs.append((src, tgt))
    return pairs

def batches_of(pairs, bs, rng, pad_id):
    idx = list(range(len(pairs))); rng.shuffle(idx)
    mega = [sorted(idx[i:i + bs * 50], key=lambda j: len(pairs[j][1])) for i in range(0, len(idx), bs * 50)]
    bl = [m[i:i + bs] for m in mega for i in range(0, len(m), bs)]
    rng.shuffle(bl)
    for b in bl:
        S = max(len(pairs[j][0]) for j in b); T = max(len(pairs[j][1]) for j in b)
        inp = torch.full((len(b), S), pad_id); att = torch.zeros((len(b), S), dtype=torch.long)
        lab = torch.full((len(b), T), -100)
        for r, j in enumerate(b):
            s, t = pairs[j]
            inp[r, :len(s)] = torch.tensor(s); att[r, :len(s)] = 1; lab[r, :len(t)] = torch.tensor(t)
        yield dict(input_ids=inp.to(DEV), attention_mask=att.to(DEV), labels=lab.to(DEV))

def train_loop(model, pairs, pad_id, lr, bs, accum, epochs, patience, warmup, fp16, eval_fn, seed, ckpt):
    set_seed(seed); rng = random.Random(seed)
    model.to(DEV); model.config.use_cache = False; model.gradient_checkpointing_enable()
    opt = Adafactor(model.parameters(), lr=lr, scale_parameter=False, relative_step=False,
                    warmup_init=False, clip_threshold=1.0, weight_decay=1e-3)
    steps_ep = math.ceil(len(pairs) / bs / accum); total = max(1, steps_ep * epochs)
    sched = LambdaLR(opt, lambda s: (s + 1) / warmup if s < warmup else max(0.05, (total - s) / max(1, total - warmup)))
    scaler = torch.amp.GradScaler("cuda", enabled=fp16)
    best, bad, hist = -1e9, 0, []
    for ep in range(epochs):
        model.train(); t0 = time.time(); losses = []
        for bi, batch in enumerate(tqdm(batches_of(pairs, bs, rng, pad_id), total=math.ceil(len(pairs) / bs), desc=f"epoch {ep + 1}/{epochs}", leave=False)):
            with torch.autocast("cuda", dtype=torch.float16, enabled=fp16):
                loss = model(**batch).loss / accum
            scaler.scale(loss).backward(); losses.append(loss.item() * accum)
            if (bi + 1) % accum == 0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        model.gradient_checkpointing_disable(); model.config.use_cache = True
        score = eval_fn(model)
        model.config.use_cache = False; model.gradient_checkpointing_enable()
        hist.append(dict(epoch=ep + 1, loss=float(np.mean(losses)), dev=score, min=(time.time() - t0) / 60))
        print(hist[-1])
        if score > best:
            best, bad = score, 0; torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(torch.load(ckpt, map_location=DEV)); os.remove(ckpt)
    model.gradient_checkpointing_disable(); model.config.use_cache = True
    return model, hist

@torch.no_grad()
def nllb_generate(model, engs, lmp_id, extra, beams):
    model.eval()
    srcs = [[ENG_ID] + encode_plain(e)[: CFG["MAX_SRC_TOK"] - 2] + [tok.eos_token_id] for e in engs]
    order = sorted(range(len(srcs)), key=lambda i: -len(srcs[i])); res = [None] * len(srcs)
    for i in tqdm(range(0, len(order), CFG["GEN_BS"]), desc=f"generate (beam {beams})", leave=False):
        b = order[i:i + CFG["GEN_BS"]]; S = max(len(srcs[j]) for j in b)
        inp = torch.full((len(b), S), tok.pad_token_id); att = torch.zeros((len(b), S), dtype=torch.long)
        for r, j in enumerate(b):
            inp[r, :len(srcs[j])] = torch.tensor(srcs[j]); att[r, :len(srcs[j])] = 1
        with torch.autocast("cuda", dtype=torch.float16, enabled=CFG["FP16"]):
            out = model.generate(input_ids=inp.to(DEV), attention_mask=att.to(DEV), forced_bos_token_id=lmp_id,
                                 num_beams=beams, max_new_tokens=CFG["MAX_TGT_TOK"])
        for r, j in enumerate(b):
            res[j] = ids_to_text(out[r].tolist(), extra, skip={lmp_id})
    return res

def run_nllb(system, seed):
    d = os.path.join(OUT_RUNS, "runs", f"{system}_seed{seed}")
    if os.path.exists(os.path.join(d, "test_hyp.txt")):
        print("skip (done):", d); return
    os.makedirs(d, exist_ok=True)
    set_seed(seed)
    model, lmp_id, mark_ids = setup_nllb(system)
    extra = {v: k for k, v in mark_ids.items()}
    pairs = build_pairs(train, system, lmp_id, mark_ids)
    devs = dev.sample(min(CFG["DEV_EVAL_N"], len(dev)), random_state=0)
    dev_refs = [target_of(l, system) for l in devs.lmp]
    def eval_fn(m):
        hyps = []
        hyps = nllb_generate(m, list(devs.eng), lmp_id, extra, beams=1)
        return chrf(hyps, dev_refs)
    model, hist = train_loop(model, pairs, tok.pad_token_id, CFG["NLLB_LR"], CFG["NLLB_BS"], CFG["NLLB_ACCUM"],
                             CFG["NLLB_EPOCHS"], CFG["NLLB_PATIENCE"], CFG["NLLB_WARMUP"], CFG["FP16"],
                             eval_fn, seed, "/content/ckpt_nllb.pt")
    t0 = time.time()
    hyps = nllb_generate(model, list(test.eng), lmp_id, extra, beams=CFG["BEAMS"])
    save_lines(os.path.join(d, "test_hyp.txt"), hyps)
    json.dump(dict(system=system, seed=seed, history=hist, gen_min=(time.time() - t0) / 60, cfg=CFG),
              open(os.path.join(d, "meta.json"), "w"), indent=1, default=str)
    del model; gc.collect(); torch.cuda.empty_cache()

for seed in CFG["NLLB_SEEDS"]:
    for system in CFG["SYSTEMS"]:
        print(f"\n=== NLLB {system} seed {seed} ===")
        run_nllb(system, seed)


=== NLLB direct seed 1 ===
skip (done): /content/drive/MyDrive/Limbum Research/limbum_tone_unicode/runs/direct_seed1

=== NLLB explicit seed 1 ===
skip (done): /content/drive/MyDrive/Limbum Research/limbum_tone_unicode/runs/explicit_seed1

=== NLLB stripped seed 1 ===
skip (done): /content/drive/MyDrive/Limbum Research/limbum_tone_unicode/runs/stripped_seed1

=== NLLB direct seed 2 ===


epoch 1/8:   0%|          | 0/883 [00:00<?, ?it/s]

generate (beam 1):   0%|          | 0/19 [00:00<?, ?it/s]

{'epoch': 1, 'loss': 3.2274337939470836, 'dev': 20.820397513188837, 'min': 7.143274835745493}


epoch 2/8:   0%|          | 0/883 [00:00<?, ?it/s]

generate (beam 1):   0%|          | 0/19 [00:00<?, ?it/s]

... (146 outputs left out)

epoch 8/8:   0%|          | 0/883 [00:00<?, ?it/s]

generate (beam 1):   0%|          | 0/19 [00:00<?, ?it/s]

{'epoch': 8, 'loss': 0.655932157337328, 'dev': 39.991621691492654, 'min': 6.180065973599752}


generate (beam 4):   0%|          | 0/56 [00:00<?, ?it/s]

## Step 3 — Tone restorers (for the Stripped system)
* **Lexical**: most frequent toned spelling of each stripped word in the training targets.
* **ByT5**: byte-level seq2seq, stripped Limbum chunk → toned chunk.
* **ByT5-src**: same, but the English source sentence is also in the input (it can
  disambiguate e.g. *come* vs *be*).

A **consistency guard** guarantees restorers only *add* tone: any output word whose stripped
form differs from the input word is replaced by the input word (fallback rate is reported).

In [23]:
def apply_case(src_core, toned_lower):
    return toned_lower[:1].upper() + toned_lower[1:] if src_core[:1].isupper() else toned_lower

def restore_lexical(text):
    out = []
    for t in strip_tones(text).split():
        pre, core, post = split_affix(t)
        k = key(core)
        if core and k in lex:
            core = apply_case(core, lex[k].most_common(1)[0][0])
        out.append(pre + core + post)
    return " ".join(out)

def guard(inp, outp):
    """Keep restorer output only where it differs from input by tone marks alone."""
    iw, ow = inp.split(), outp.split()
    if len(iw) == len(ow) and all(strip_tones(o) == nfc(i) for i, o in zip(iw, ow)):
        return outp, 0
    sm = difflib.SequenceMatcher(None, [nfc(w) for w in iw], [strip_tones(w) for w in ow], autojunk=False)
    res, fb = list(iw), len(iw)
    for a, b, n in sm.get_matching_blocks():
        for k in range(n):
            res[a + k] = ow[b + k]; fb -= 1
    return " ".join(res), fb

def chunks(text, n):
    w = text.split()
    return [" ".join(w[i:i + n]) for i in range(0, len(w), n)] or [""]

def byt5_input(stripped_chunk, eng, src_aware):
    return f"en: {eng[:CFG['ENG_MAX_CHARS']]} | lmp: {stripped_chunk}" if src_aware else stripped_chunk

def restorer_examples(df, src_aware):
    ex = []
    for e, l in zip(df.eng, df.lmp):
        for c in chunks(l, CFG["CHUNK_WORDS"]):
            ex.append((byt5_input(strip_tones(c), e, src_aware), c))
    return ex

@torch.no_grad()
def byt5_restore(model, btok, texts, engs, src_aware):
    model.eval()
    jobs = []
    texts = [strip_tones(t) for t in texts]          # restorers only ever see tone-free input
    for i, (t, e) in enumerate(zip(texts, engs)):
        for c in chunks(t, CFG["CHUNK_WORDS"]):
            jobs.append((i, c, byt5_input(c, e, src_aware)))
    order = sorted(range(len(jobs)), key=lambda j: -len(jobs[j][2].encode()))
    outs = [None] * len(jobs)
    for s in tqdm(range(0, len(order), CFG["RESTORE_BS"]), desc="restore tones", leave=False):
        b = order[s:s + CFG["RESTORE_BS"]]
        enc = btok([jobs[j][2] for j in b], return_tensors="pt", padding=True, truncation=True,
                   max_length=CFG["BYT5_MAX_IN"]).to(DEV)
        gen = model.generate(**enc, max_new_tokens=CFG["BYT5_MAX_OUT"], num_beams=1)
        for j, g in zip(b, btok.batch_decode(gen, skip_special_tokens=True)):
            outs[j] = g
    per = defaultdict(list); fb_total = words = 0
    for (i, c, _), o in zip(jobs, outs):
        g, fb = guard(c, nfc(clean_ws(o)))
        per[i].append(g); fb_total += fb; words += len(c.split())
    return [" ".join(per[i]) for i in range(len(texts))], 100 * fb_total / max(1, words)

def train_byt5(src_aware, seed):
    from transformers import AutoTokenizer as AT, AutoModelForSeq2SeqLM as AM
    name = f"byt5{'_src' if src_aware else ''}_seed{seed}"
    d = os.path.join(OUT_RUNS, "restorers", name); os.makedirs(d, exist_ok=True)
    wpath = os.path.join(d, "model")
    btok = AT.from_pretrained(CFG["BYT5_MODEL"])
    if os.path.exists(os.path.join(wpath, "config.json")):
        return AM.from_pretrained(wpath).to(DEV), btok, name
    model = AM.from_pretrained(CFG["BYT5_MODEL"])
    ex = restorer_examples(train, src_aware)
    pairs = [(btok(a, truncation=True, max_length=CFG["BYT5_MAX_IN"]).input_ids,
              btok(b, truncation=True, max_length=CFG["BYT5_MAX_OUT"]).input_ids) for a, b in ex]
    devs = dev.sample(min(CFG["DEV_EVAL_N"], len(dev)), random_state=0)
    dev_in = [strip_tones(l) for l in devs.lmp]
    def eval_fn(m):
        hyps, _ = byt5_restore(m, btok, dev_in, list(devs.eng), src_aware)
        return tone_eval(hyps, list(devs.lmp))["TAcc"]
    model, hist = train_loop(model, pairs, btok.pad_token_id, CFG["BYT5_LR"], CFG["BYT5_BS"], CFG["BYT5_ACCUM"],
                             CFG["BYT5_EPOCHS"], CFG["BYT5_PATIENCE"], CFG["BYT5_WARMUP"], False,  # fp32 (T5 is unstable in fp16)
                             eval_fn, seed, "/content/ckpt_byt5.pt")
    model.save_pretrained(wpath); json.dump(hist, open(os.path.join(d, "history.json"), "w"), indent=1)
    return model, btok, name

## Metrics (defined before restorer training because the dev criterion uses tone accuracy)

In [10]:
def tone_eval(hyps, refs, per_sentence=False):
    C = Counter(); sent = []
    for h, r in zip(hyps, refs):
        hi, ri = word_items(nfc(h)), word_items(nfc(r))
        hk, rk = [key(w) for w, _ in hi], [key(w) for w, _ in ri]
        sm = difflib.SequenceMatcher(None, hk, rk, autojunk=False)
        s = Counter(ref=len(ri))
        for a, b, n in sm.get_matching_blocks():
            for k in range(n):
                hw, (rw, fin) = hi[a + k][0], ri[b + k]
                ok = nfc(hw).lower() == nfc(rw).lower(); kk = rk[b + k]
                cats = ["all", "final" if fin else "nonfinal",
                        "oov" if kk not in lex_seen else ("amb" if kk in lex_amb else "unamb")]
                if count_marks(rw) > 0: cats.append("marked")
                for c in cats:
                    s[f"m_{c}"] += 1; s[f"c_{c}"] += ok
        s["hyp_marks"] = sum(count_marks(w) for w, _ in hi); s["ref_marks"] = sum(count_marks(w) for w, _ in ri)
        C.update(s); sent.append(s)
    pct = lambda a, b: 100 * a / b if b else float("nan")
    out = dict(TAcc=pct(C["c_all"], C["m_all"]), TAccM=pct(C["c_marked"], C["m_marked"]),
               TAccAmb=pct(C["c_amb"], C["m_amb"]), TAccUnamb=pct(C["c_unamb"], C["m_unamb"]),
               TAccOOV=pct(C["c_oov"], C["m_oov"]), TAccFinal=pct(C["c_final"], C["m_final"]),
               TAccNonfinal=pct(C["c_nonfinal"], C["m_nonfinal"]), Coverage=pct(C["m_all"], C["ref"]),
               MarkRatio=C["hyp_marks"] / C["ref_marks"] if C["ref_marks"] else float("nan"),
               N_all=C["m_all"], N_marked=C["m_marked"], N_amb=C["m_amb"], N_unamb=C["m_unamb"],
               N_oov=C["m_oov"], N_final=C["m_final"], N_nonfinal=C["m_nonfinal"])
    return (out, sent) if per_sentence else out

def full_eval(hyps, refs):
    hs, rs = [strip_tones(h) for h in hyps], [strip_tones(r) for r in refs]
    m = dict(chrfT=chrf(hyps, refs), chrfS=chrf(hs, rs), bleuT=BLEU_M.corpus_score(hyps, [refs]).score)
    m["Delta"] = m["chrfS"] - m["chrfT"]
    m.update(tone_eval(hyps, refs))
    return m

## Step 3 (run) — restorers: oracle evaluation + application to Stripped-system outputs

In [25]:
test_refs = list(test.lmp); test_engs = list(test.eng)
gold_stripped = [strip_tones(r) for r in test_refs]
RESTORED = {}   # (restorer, nllb_seed) -> hyps
ORACLE = {}     # restorer -> metrics on gold stripped input
FALLBACK = {}

ORACLE["lexical"] = full_eval([restore_lexical(s) for s in gold_stripped], test_refs)
for s in CFG["NLLB_SEEDS"]:
    p = os.path.join(OUT_RUNS, "runs", f"stripped_seed{s}", "test_hyp.txt")
    if os.path.exists(p):
        RESTORED[("lexical", s)] = [restore_lexical(h) for h in load_lines(p)]

for rseed in CFG["RESTORER_SEEDS"]:
    for src_aware in [False, True]:
        model, btok, name = train_byt5(src_aware, rseed)
        rname = "byt5_src" if src_aware else "byt5"
        cache = os.path.join(OUT_RUNS, "restorers", name, "oracle_hyp.txt")
        if os.path.exists(cache):
            oh = load_lines(cache); fbo = json.load(open(cache + ".fb.json"))["fb"]
        else:
            oh, fbo = byt5_restore(model, btok, gold_stripped, test_engs, src_aware)
            save_lines(cache, oh); json.dump(dict(fb=fbo), open(cache + ".fb.json", "w"))
        ORACLE[rname] = full_eval(oh, test_refs); FALLBACK[(rname, "oracle")] = fbo
        for s in CFG["NLLB_SEEDS"]:
            p = os.path.join(OUT_RUNS, "runs", f"stripped_seed{s}", "test_hyp.txt")
            c2 = os.path.join(OUT_RUNS, "restorers", name, f"on_stripped_seed{s}.txt")
            if not os.path.exists(p): continue
            if os.path.exists(c2):
                RESTORED[(rname, s)] = load_lines(c2)
            else:
                rh, fb = byt5_restore(model, btok, load_lines(p), test_engs, src_aware)
                save_lines(c2, rh); RESTORED[(rname, s)] = rh; FALLBACK[(rname, s)] = fb
        del model; gc.collect(); torch.cuda.empty_cache()
print(pd.DataFrame(ORACLE).T.round(2))

restore tones:   0%|          | 0/55 [00:00<?, ?it/s]

restore tones:   0%|          | 0/53 [00:00<?, ?it/s]

restore tones:   0%|          | 0/55 [00:00<?, ?it/s]

restore tones:   0%|          | 0/53 [00:00<?, ?it/s]

          chrfT  chrfS  bleuT  Delta   TAcc  TAccM  TAccAmb  TAccUnamb  \
lexical   83.70  100.0  70.66  16.30  85.70  71.30    65.10      97.60   
byt5      94.88  100.0  90.54   5.12  95.59  93.35    91.34      98.45   
byt5_src  95.06  100.0  90.89   4.94  95.75  93.41    91.72      98.45   

          TAccOOV  TAccFinal  TAccNonfinal  Coverage  MarkRatio    N_all  \
lexical     23.65      86.81         85.64     100.0       0.83  26002.0   
byt5        54.36      95.03         95.62     100.0       1.00  26002.0   
byt5_src    57.68      95.44         95.77     100.0       1.00  26002.0   

          N_marked   N_amb  N_unamb  N_oov  N_final  N_nonfinal  
lexical    10293.0  8969.0  16792.0  241.0   1448.0     24554.0  
byt5       10293.0  8969.0  16792.0  241.0   1448.0     24554.0  
byt5_src   10293.0  8969.0  16792.0  241.0   1448.0     24554.0  


## Step 4 — Evaluate every system, aggregate over seeds, paired bootstrap

In [26]:
SYS_LABEL = {"Direct": "Direct", "Explicit": "Explicit tone tokens", "Strip": "Stripped (no restoration)",
             "StripLex": "Stripped + lexical restore", "StripByT": "Stripped + ByT5 restore",
             "StripByTSrc": "Stripped + source-aware ByT5"}
HYPS = defaultdict(dict)   # sys -> seed -> hyps
for s in CFG["NLLB_SEEDS"]:
    for sysname, folder in [("Direct", "direct"), ("Explicit", "explicit"), ("Strip", "stripped")]:
        p = os.path.join(OUT_RUNS, "runs", f"{folder}_seed{s}", "test_hyp.txt")
        if os.path.exists(p): HYPS[sysname][s] = load_lines(p)
    for rname, sysname in [("lexical", "StripLex"), ("byt5", "StripByT"), ("byt5_src", "StripByTSrc")]:
        if (rname, s) in RESTORED: HYPS[sysname][s] = RESTORED[(rname, s)]

rows = []
for sysname, per_seed in HYPS.items():
    for s, h in per_seed.items():
        rows.append(dict(system=sysname, seed=s, **full_eval(h, test_refs)))
per_seed_df = pd.DataFrame(rows)
per_seed_df.to_csv(os.path.join(OUT_RUNS, "results_per_seed.csv"), index=False)
agg = per_seed_df.drop(columns="seed").groupby("system").agg(["mean", "std"])
display(per_seed_df.groupby("system").mean(numeric_only=True).round(2))

# --- paired bootstrap vs Direct (same resampled sentence indices for both systems) ---
def sent_stats_chrf(h, r): return np.array(CHRF_M._extract_corpus_statistics(h, [r]))
def boot_compare(hA, hB, refs, n, seed=0):
    rng = np.random.default_rng(seed)
    sa, sb = sent_stats_chrf(hA, refs), sent_stats_chrf(hB, refs)
    _, ta = tone_eval(hA, refs, per_sentence=True); _, tb = tone_eval(hB, refs, per_sentence=True)
    ca = np.array([[t["c_all"], t["m_all"]] for t in ta], float); cb = np.array([[t["c_all"], t["m_all"]] for t in tb], float)
    N = len(refs); dchrf, dtacc = [], []
    for _ in range(n):
        idx = rng.integers(0, N, N)
        fa = CHRF_M._compute_score_from_stats(sa[idx].sum(0).tolist()).score
        fb = CHRF_M._compute_score_from_stats(sb[idx].sum(0).tolist()).score
        dchrf.append(fb - fa)
        A, B = ca[idx].sum(0), cb[idx].sum(0)
        dtacc.append(100 * (B[0] / B[1] - A[0] / A[1]))
    dchrf, dtacc = np.array(dchrf), np.array(dtacc)
    return dict(d_chrfT=float(np.mean(dchrf)), p_chrfT=float(np.mean(dchrf <= 0)),
                ci_chrfT=(float(np.percentile(dchrf, 2.5)), float(np.percentile(dchrf, 97.5))),
                d_TAcc=float(np.mean(dtacc)), p_TAcc=float(np.mean(dtacc <= 0)),
                ci_TAcc=(float(np.percentile(dtacc, 2.5)), float(np.percentile(dtacc, 97.5))))

sig = []
for sysname in HYPS:
    if sysname == "Direct": continue
    for s in CFG["NLLB_SEEDS"]:
        if s in HYPS[sysname] and s in HYPS["Direct"]:
            sig.append(dict(system=sysname, seed=s, **boot_compare(HYPS["Direct"][s], HYPS[sysname][s],
                                                                 test_refs, CFG["BOOTSTRAP_N"], seed=s)))
sig_df = pd.DataFrame(sig); sig_df.to_csv(os.path.join(OUT_RUNS, "significance_vs_direct.csv"), index=False)
display(sig_df)

,seed,chrfT,chrfS,bleuT,Delta,TAcc,TAccM,TAccAmb,TAccUnamb,TAccOOV,...,TAccNonfinal,Coverage,MarkRatio,N_all,N_marked,N_amb,N_unamb,N_oov,N_final,N_nonfinal
system,,,,,,,,,,,,,,,,,,,,,
Direct,2.0,44.28,46.18,26.58,1.90,94.73,92.89,88.11,98.45,38.39,...,94.77,49.64,1.03,12906.33,5153.33,4502.67,8379.33,24.33,715.00,12191.33
Explicit,2.0,44.37,46.30,26.70,1.93,94.62,93.12,87.66,98.52,34.93,...,94.65,49.48,1.09,12864.67,5163.00,4515.00,8331.00,18.67,700.33,12164.33
Strip,2.0,30.00,46.97,9.56,16.98,60.12,0.00,53.15,64.10,0.00,...,60.77,50.36,0.00,13095.67,5222.67,4607.00,8462.67,26.00,719.67,12376.00
StripByT,2.0,44.86,46.97,26.89,2.12,93.94,91.87,85.98,98.37,62.64,...,93.92,50.36,0.99,13095.67,5222.67,4607.00,8462.67,26.00,719.67,12376.00
StripByTSrc,2.0,44.88,46.97,27.00,2.09,94.00,91.66,86.11,98.41,53.70,...,93.97,50.36,0.98,13095.67,5222.67,4607.00,8462.67,26.00,719.67,12376.00
StripLex,2.0,41.11,46.97,20.72,5.87,86.01,70.99,64.19,98.16,0.00,...,85.83,50.36,0.83,13095.67,5222.67,4607.00,8462.67,26.00,719.67,12376.00


,system,seed,d_chrfT,p_chrfT,ci_chrfT,d_TAcc,p_TAcc,ci_TAcc
0,Explicit,1,0.071227,0.404,"(-0.4334517668581766, 0.6096773246707827)",-0.330226,0.943,"(-0.735858902857884, 0.08119359312244823)"
1,Explicit,2,0.063189,0.397,"(-0.4248591023603122, 0.5751931867358138)",-0.196708,0.842,"(-0.5934726453955838, 0.21815472772151506)"
2,Explicit,3,0.140680,0.323,"(-0.40087269832298844, 0.6680888231064781)",0.212537,0.148,"(-0.19510426010790002, 0.6172017376888501)"
3,Strip,1,-14.362572,1.000,"(-15.011811321370867, -13.679084146864081)",-34.564899,1.000,"(-35.439150621461664, -33.747349511763154)"
4,Strip,2,-14.487986,1.000,"(-15.124930783582455, -13.825641573287607)",-34.654939,1.000,"(-35.493038326137224, -33.81553478637862)"
5,Strip,3,-13.978986,1.000,"(-14.677136416769201, -13.33393415105322)",-34.602300,1.000,"(-35.562085782916625, -33.75799922457969)"
6,StripLex,1,-3.300638,1.000,"(-3.88335184655316, -2.7027455149473525)",-8.818217,1.000,"(-9.528859886339333, -8.125063490863184)"
7,StripLex,2,-3.474320,1.000,"(-4.028895039849859, -2.8916312000261373)",-8.665142,1.000,"(-9.31393397454023, -8.003361388846907)"
8,StripLex,3,-2.745038,1.000,"(-3.307626142787388, -2.1930314382576777)",-8.685920,1.000,"(-9.373594249651624, -8.013611704870868)"
9,StripByT,1,0.382930,0.101,"(-0.19560681731286067, 0.9240822749442564)",-1.040566,1.000,"(-1.4959263884906226, -0.5682624487561592)"


## Step 5 — Export LaTeX tables, `\res{}` macros, figure, error list, human-evaluation sheet

In [27]:
PA = os.path.join(OUT_RUNS if CFG["QUICK"] else OUT, "paper_assets")
for sub in ["tables", "results", "figs"]: os.makedirs(os.path.join(PA, sub), exist_ok=True)
fmt = lambda x, d=1: "--" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:.{d}f}"
macros = {}
def M(k, v, d=1): macros[k] = fmt(v, d) if isinstance(v, (float, int, np.floating)) else str(v)

for k, v in AUDIT.items(): M("Audit" + k, v, (2 if k in ("MarksPerWord", "UnkPerHundredNFC") or k.startswith("Fert") else 1) if isinstance(v, float) else 0)
M("NTrain", len(train), 0); M("NDev", len(dev), 0); M("NTest", len(test), 0)
M("NTestNT", int((test.source == "nt").sum()), 0); M("NTestDict", int((test.source == "dict").sum()), 0)
M("NSeeds", len(CFG["NLLB_SEEDS"]), 0); M("SplitHow", split_how.split(" (")[0])

COLS = [("chrfT", "chrF++$_T$"), ("chrfS", "chrF++$_S$"), ("Delta", "$\\Delta$"), ("bleuT", "BLEU$_T$"),
        ("TAcc", "TAcc"), ("TAccM", "TAcc$_M$"), ("TAccAmb", "TAcc$_{amb}$"), ("MarkRatio", "MR"), ("Coverage", "Cov.")]
best_tacc = per_seed_df.groupby("system")["TAccM"].mean().idxmax() if len(per_seed_df) else None
lines = ["\\begin{tabular}{l" + "r" * len(COLS) + "}", "\\toprule",
         "System & " + " & ".join(c for _, c in COLS) + " \\\\", "\\midrule"]
for sysname in ["Direct", "Explicit", "Strip", "StripLex", "StripByT", "StripByTSrc"]:
    if sysname not in agg.index: continue
    cells = []
    for c, _ in COLS:
        mu, sd = agg.loc[sysname, (c, "mean")], agg.loc[sysname, (c, "std")]
        d = 2 if c == "MarkRatio" else 1
        cell = fmt(mu, d) + (f"{{\\scriptsize$\\pm${fmt(sd, d)}}}" if len(CFG["NLLB_SEEDS"]) > 1 and not math.isnan(sd) else "")
        if c in ("chrfT", "TAcc") and sysname != "Direct" and len(sig_df):
            ps = sig_df[sig_df.system == sysname][f"p_{c}"] if f"p_{c}" in sig_df else []
            if len(ps) and (ps < 0.05).all(): cell += "$^\\dagger$"
        cells.append(cell)
        M(f"{sysname}{c[0].upper() + c[1:]}", mu, d)
    name = SYS_LABEL[sysname] + (" $\\star$" if sysname == best_tacc else "")
    lines.append(name + " & " + " & ".join(cells) + " \\\\")
    if sysname == "Explicit": lines.append("\\midrule")
lines += ["\\bottomrule", "\\end{tabular}"]
open(os.path.join(PA, "tables/main_results.tex"), "w").write("\n".join(lines))

ocols = [("TAcc", "TAcc"), ("TAccM", "TAcc$_M$"), ("TAccAmb", "TAcc$_{amb}$"), ("TAccOOV", "TAcc$_{oov}$"), ("chrfT", "chrF++$_T$")]
ol = ["\\begin{tabular}{l" + "r" * (len(ocols) + 1) + "}", "\\toprule",
      "Restorer & " + " & ".join(c for _, c in ocols) + " & Fallback\\% \\\\", "\\midrule"]
for r, lab in [("lexical", "Lexical (most frequent)"), ("byt5", "ByT5"), ("byt5_src", "Source-aware ByT5")]:
    if r not in ORACLE: continue
    fbv = FALLBACK.get((r, "oracle"), 0.0)
    ol.append(lab + " & " + " & ".join(fmt(ORACLE[r][c]) for c, _ in ocols) + f" & {fmt(fbv)} \\\\")
    for c, _ in ocols: M(f"Oracle{ {'lexical': 'Lex', 'byt5': 'ByT', 'byt5_src': 'ByTSrc'}[r] }{c[0].upper() + c[1:]}", ORACLE[r][c])
ol += ["\\bottomrule", "\\end{tabular}"]
open(os.path.join(PA, "tables/oracle_restoration.tex"), "w").write("\n".join(ol))

CATS = [("TAccUnamb", "Unambiguous"), ("TAccAmb", "Ambiguous"), ("TAccOOV", "Unseen (OOV)"),
        ("TAccNonfinal", "Phrase-medial"), ("TAccFinal", "Phrase-final")]
mean_df = per_seed_df.groupby("system").mean(numeric_only=True)
ORDER = [x for x in ["Direct", "Explicit", "Strip", "StripLex", "StripByT", "StripByTSrc"] if x in mean_df.index]
mean_df = mean_df.loc[ORDER]
al = ["\\begin{tabular}{l" + "r" * len(CATS) + "}", "\\toprule",
      "System & " + " & ".join(c for _, c in CATS) + " \\\\", "\\midrule"]
for sysname in mean_df.index:
    al.append(SYS_LABEL[sysname] + " & " + " & ".join(fmt(mean_df.loc[sysname, c]) for c, _ in CATS) + " \\\\")
    for c, _ in CATS: M(f"{sysname}{c}", mean_df.loc[sysname, c])
if "Direct" in mean_df.index:
    NK = {"TAccUnamb": "N_unamb", "TAccAmb": "N_amb", "TAccOOV": "N_oov", "TAccNonfinal": "N_nonfinal", "TAccFinal": "N_final"}
    al.append("\\midrule")
    al.append("\\textit{\\# matched ref.\\ words (Direct)} & " + " & ".join(
        str(int(round(mean_df.loc["Direct", NK[c]]))) for c, _ in CATS) + " \\\\")
al += ["\\bottomrule", "\\end{tabular}"]
open(os.path.join(PA, "tables/error_breakdown.tex"), "w").write("\n".join(al))

au = ["\\begin{tabular}{lr}", "\\toprule", "Statistic & Value \\\\", "\\midrule",
      f"Words carrying $\\geq$1 tone mark (NT / dictionary) & {fmt(AUDIT['WordsMarkedPctNT'])}\\% / {fmt(AUDIT['WordsMarkedPctDICT'])}\\% \\\\",
      f"Tone marks per word & {fmt(AUDIT['MarksPerWord'], 2)} \\\\",
      f"Acute / grave / macron occurrences & {AUDIT['AcuteCount']} / {AUDIT['GraveCount']} / {AUDIT['MacronCount']} \\\\",
      f"Stripped types with $\\geq$2 toned spellings (all / freq.$\\geq$5) & {fmt(AUDIT['AmbTypesPct'])}\\% / {fmt(AUDIT['AmbTypesPctFreqFive'])}\\% \\\\",
      f"Running words that are tone-ambiguous & {fmt(AUDIT['AmbTokensPct'])}\\% \\\\", "\\midrule",
      f"NLLB round trip exact, NFC input & {fmt(AUDIT['RTExactNFC'])}\\% \\\\",
      f"Tone-mark survival, NFC / NFD input & {fmt(AUDIT['RTMarkSurvNFC'])}\\% / {fmt(AUDIT['RTMarkSurvNFD'])}\\% \\\\",
      f"\\texttt{{<unk>}} per 100 tokens (NFC) & {fmt(AUDIT['UnkPerHundredNFC'], 2)} \\\\",
      f"Tokens per word: toned / stripped / explicit & {fmt(AUDIT['FertToned'], 2)} / {fmt(AUDIT['FertStripped'], 2)} / {fmt(AUDIT['FertExplicit'], 2)} \\\\",
      f"Explicit encoding round trip exact & {fmt(AUDIT['RTExactExplicit'])}\\% \\\\",
      "\\bottomrule", "\\end{tabular}"]
open(os.path.join(PA, "tables/audit.tex"), "w").write("\n".join(au))

with open(os.path.join(PA, "results/numbers.tex"), "w") as f:
    f.write("% auto-generated by the experiment notebook — do not edit by hand\n")
    for k, v in macros.items():
        safe = re.sub(r"[^A-Za-z]", "", k)
        f.write(f"\\expandafter\\def\\csname res@{safe}\\endcsname{{{v}}}\n")
print(len(macros), "macros written")

# figure: tone accuracy by word category
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.4, 2.8))
syss = list(mean_df.index); w = 0.8 / max(1, len(syss)); x = np.arange(len(CATS))
for i, s in enumerate(syss):
    ax.bar(x + i * w, [mean_df.loc[s, c] for c, _ in CATS], w, label=SYS_LABEL[s])
ax.set_xticks(x + w * (len(syss) - 1) / 2); ax.set_xticklabels([l for _, l in CATS], fontsize=8)
ax.set_ylabel("Tone accuracy (%)"); ax.legend(fontsize=6, ncol=2, frameon=False); ax.set_ylim(0, 100)
plt.tight_layout(); plt.savefig(os.path.join(PA, "figs/tacc_breakdown.pdf")); plt.close()

# word-level error list on ambiguous words (for qualitative analysis section)
def error_rows(sysname, seed):
    rows = []
    for i, (h, r) in enumerate(zip(HYPS[sysname][seed], test_refs)):
        hi, ri = word_items(nfc(h)), word_items(nfc(r))
        sm = difflib.SequenceMatcher(None, [key(w) for w, _ in hi], [key(w) for w, _ in ri], autojunk=False)
        for a, b, n in sm.get_matching_blocks():
            for k in range(n):
                hw, rw = hi[a + k][0], ri[b + k][0]
                if key(rw) in lex_amb and nfc(hw).lower() != nfc(rw).lower():
                    rows.append(dict(system=sysname, sent=i, english=test_engs[i], ref_word=rw, hyp_word=hw,
                                     train_variants="; ".join(f"{f}×{c}" for f, c in lex[key(rw)].most_common())))
    return rows
s0 = CFG["NLLB_SEEDS"][0]
errs = sum([error_rows(s, s0) for s in HYPS if s0 in HYPS[s]], [])
pd.DataFrame(errs).to_csv(os.path.join(OUT_RUNS, "ambiguous_word_errors.csv"), index=False)

# blind A/B sheet for Wimbum speakers: Direct vs best tone system, sentences containing ambiguous words
if best_tacc and best_tacc != "Direct" and s0 in HYPS.get(best_tacc, {}):
    cand = [i for i, r in enumerate(test_refs) if any(key(w) in lex_amb for w, _ in word_items(r))]
    rng = random.Random(0); rng.shuffle(cand); cand = cand[: CFG["HUMAN_EVAL_N"]]
    sheet, keyrows = [], []
    for n, i in enumerate(cand):
        a, b = HYPS["Direct"][s0][i], HYPS[best_tacc][s0][i]; flip = rng.random() < 0.5
        A, B = (b, a) if flip else (a, b)
        sheet.append(dict(item=n + 1, english=test_engs[i], output_A=A, output_B=B,
                          A_tone_errors_that_change_meaning="", B_tone_errors_that_change_meaning="",
                          preferred_A_B_or_same="", comments=""))
        keyrows.append(dict(item=n + 1, test_index=i, A=best_tacc if flip else "Direct", B="Direct" if flip else best_tacc))
    pd.DataFrame(sheet).to_csv(os.path.join(OUT, "human_eval/sheet_for_annotators.csv"), index=False)
    pd.DataFrame(keyrows).to_csv(os.path.join(OUT, "human_eval/KEY_do_not_share.csv"), index=False)

import shutil
shutil.make_archive(os.path.join(OUT_RUNS, "paper_assets"), "zip", PA)
print("Done. Copy paper_assets/ next to the .tex file (or unzip paper_assets.zip there) and recompile.")

119 macros written
Done. Copy paper_assets/ next to the .tex file (or unzip paper_assets.zip there) and recompile.


In [28]:
# Legacy-vs-Unicode comparison (CPU only, ~2-3 min, no training).
# Run after: pip cell -> Config -> Drive -> Utilities -> Step 0 -> Step 1a -> Metrics cell.
import os, re, unicodedata as ud
import numpy as np, pandas as pd
from sacrebleu.metrics import CHRF, BLEU

CHRF_M, BLEU_M = CHRF(word_order=2), BLEU()
def chrf(h, r): return CHRF_M.corpus_score(h, [r]).score       # used by full_eval

UNI_RUNS = os.path.join(OUT, "runs")
LEGACY_RUNS = os.path.join(os.path.dirname(OUT.rstrip("/")), "limbum_tone", "runs")   # edit if your old folder differs
assert os.path.isdir(LEGACY_RUNS), f"Legacy runs folder not found: {LEGACY_RUNS}"
SEEDS, SYSTEMS = [1, 2, 3], ["direct", "explicit"]

# ---- identical conversion to the data pipeline ----
GRAVE, CIRC = "\u0300", "\u0302"
TONE_GLYPH = {"è": GRAVE, "È": GRAVE, "ì": GRAVE, "Ì": GRAVE,
              "ê": CIRC, "Ê": CIRC, "â": CIRC, "Â": CIRC, "î": CIRC, "Î": CIRC, "û": CIRC, "Û": CIRC}
LETTER = {"á": "ɛ", "Á": "Ɛ", "ú": "ʉ", "Ú": "Ʉ", "ñ": "ŋ", "Ñ": "Ŋ", "ï": "i", "Ï": "I"}
VOWELS = set("aeiouɛʉAEIOUƐɄ")
QUOTES = {"“": '"', "”": '"', "‘": '"', "’": '"', "ʼ": "'"}
TM = "\u0300\u0302\u030C"
unattached = 0

def legacy_to_unicode(s):
    global unattached
    out = []
    for ch in s:
        if ch in TONE_GLYPH:
            j = len(out) - 1
            while j >= 0 and ud.category(out[j]) == "Mn": j -= 1
            if j >= 0 and out[j] in VOWELS:
                out.append(TONE_GLYPH[ch]); continue
            unattached += 1
        out.append(LETTER.get(ch, ch))
    return ud.normalize("NFC", "".join(out))

def fix_marks(s):
    d = ud.normalize("NFD", "".join(QUOTES.get(c, c) for c in s))
    d = re.sub(rf"([mnŋMNŊ]) ([{TM}])", r"\1\2", d)
    d = re.sub(rf"([{TM}])\1+", r"\1", d)
    d = re.sub(rf"(^|\s)[{TM}]+", r"\1", d)
    return ud.normalize("NFC", re.sub(r"\s+", " ", d).strip())

def convert_legacy_output(s): return fix_marks(legacy_to_unicode(s))
def no_quotes(xs): return [clean_ws(x.replace('"', " ")) for x in xs]

refs = list(test.lmp)

# ---- paired bootstrap: B (Unicode-trained) minus A (legacy-trained) ----
def boot(hA, hB, refs, n=1000, seed=0):
    rng = np.random.default_rng(seed); N = len(refs)
    sa = np.array(CHRF_M._extract_corpus_statistics(hA, [refs]))
    sb = np.array(CHRF_M._extract_corpus_statistics(hB, [refs]))
    _, ta = tone_eval(hA, refs, per_sentence=True); _, tb = tone_eval(hB, refs, per_sentence=True)
    get = lambda T, c, m: np.array([[t[c], t[m]] for t in T], float)
    A_all, B_all = get(ta, "c_all", "m_all"), get(tb, "c_all", "m_all")
    A_amb, B_amb = get(ta, "c_amb", "m_amb"), get(tb, "c_amb", "m_amb")
    d = {"chrfT": [], "TAcc": [], "TAccAmb": []}
    for _ in range(n):
        i = rng.integers(0, N, N)
        d["chrfT"].append(CHRF_M._compute_score_from_stats(sb[i].sum(0).tolist()).score
                          - CHRF_M._compute_score_from_stats(sa[i].sum(0).tolist()).score)
        for k, (A, B) in {"TAcc": (A_all, B_all), "TAccAmb": (A_amb, B_amb)}.items():
            a, b = A[i].sum(0), B[i].sum(0)
            d[k].append(100 * (b[0] / b[1] - a[0] / a[1]))
    return {f"d_{k}": float(np.mean(v)) for k, v in d.items()} | {f"p_{k}": float(np.mean(np.array(v) <= 0)) for k, v in d.items()}

rows, sig = [], []
for system in SYSTEMS:
    for s in SEEDS:
        pl = os.path.join(LEGACY_RUNS, f"{system}_seed{s}", "test_hyp.txt")
        pu = os.path.join(UNI_RUNS, f"{system}_seed{s}", "test_hyp.txt")
        if not (os.path.exists(pl) and os.path.exists(pu)):
            print("missing:", pl if not os.path.exists(pl) else pu); continue
        hl = [convert_legacy_output(x) for x in load_lines(pl)]
        hu = load_lines(pu)
        assert len(hl) == len(hu) == len(refs), (len(hl), len(hu), len(refs))
        for name, h in [("legacy-trained", hl), ("unicode-trained", hu)]:
            m = full_eval(h, refs)
            m["chrfT_noquotes"] = chrf(no_quotes(h), no_quotes(refs))
            rows.append(dict(system=system, training_text=name, seed=s, **m))
        sig.append(dict(system=system, seed=s, **boot(hl, hu, refs, seed=s)))

res = pd.DataFrame(rows); sig = pd.DataFrame(sig)
cols = ["chrfT", "chrfT_noquotes", "chrfS", "bleuT", "TAcc", "TAccM", "TAccAmb", "Coverage", "MarkRatio"]
summary = res.groupby(["system", "training_text"])[cols].agg(["mean", "std"]).round(2)
print(f"Unattached legacy tone glyphs in model outputs (left unconverted): {unattached}")
display(summary)
print("\nPaired bootstrap, Unicode-trained minus legacy-trained (p = share of resamples with no gain):")
display(sig.round(3))
res.to_csv(os.path.join(OUT, "legacy_vs_unicode_per_seed.csv"), index=False)
sig.to_csv(os.path.join(OUT, "legacy_vs_unicode_significance.csv"), index=False)
print("Saved to", OUT)

Unattached legacy tone glyphs in model outputs (left unconverted): 812


chrfT       chrfT_noquotes        chrfS        \
                           mean   std           mean   std   mean   std   
system   training_text                                                    
direct   legacy-trained   43.59  0.32          43.92  0.33  45.54  0.29   
         unicode-trained  44.28  0.14          44.23  0.15  46.18  0.07   
explicit legacy-trained   42.95  0.41          43.27  0.41  44.87  0.41   
         unicode-trained  44.37  0.11          44.32  0.10  46.30  0.11   

                          bleuT         TAcc        TAccM       TAccAmb        \
                           mean   std   mean   std   mean   std    mean   std   
system   training_text                                                          
direct   legacy-trained   25.57  0.41  94.59  0.13  92.95  0.21   87.51  0.26   
         unicode-trained  26.58  0.16  94.73  0.09  92.89  0.09   88.11  0.24   
explicit legacy-trained   24.66  0.35  94.65  0.17  93.35  0.19   87.42  0.45   
         unicode-trained  26.70  0.14  94.62  0.19  93.12  0.40   87.66  0.56   

                         Coverage       MarkRatio        
                             mean   std      mean   std  
system   training_text                                   
direct   legacy-trained     49.65  0.34      1.07  0.02  
         unicode-trained    49.64  0.12      1.03  0.02  
explicit legacy-trained     48.63  0.57      1.08  0.02  
         unicode-trained    49.48  0.14      1.09  0.02


Paired bootstrap, Unicode-trained minus legacy-trained (p = share of resamples with no gain):


,system,seed,d_chrfT,d_TAcc,d_TAccAmb,p_chrfT,p_TAcc,p_TAccAmb
0,direct,1,0.386,0.065,0.472,0.080,0.368,0.164
1,direct,2,1.179,0.234,0.790,0.000,0.117,0.057
2,direct,3,0.501,0.123,0.513,0.056,0.281,0.172
3,explicit,1,1.554,-0.026,0.416,0.000,0.559,0.219
4,explicit,2,1.084,-0.291,-0.645,0.000,0.918,0.876
5,explicit,3,1.627,0.242,0.984,0.000,0.118,0.036


Saved to /content/drive/MyDrive/Limbum Research/limbum_tone_unicode


In [31]:
import re
from sacrebleu.metrics import CHRF
c = CHRF(word_order=2)
words = lambda t: [w.lower() for w, _ in word_items(t)]          # words only, no punctuation, lower-case
exact = 0; cands = []
for s in [1, 2, 3]:
    hyp = load_lines(os.path.join(OUT, f"runs/direct_seed{s}/test_hyp.txt"))
    for i, (r, h) in enumerate(zip(test.lmp, hyp)):
        rw, hw = words(r), words(h)
        if rw == hw: exact += 1; continue
        if len(rw) == len(hw) and [key(w) for w in rw] == [key(w) for w in hw]:   # same words, tone differs
            diff = [(a, b) for a, b in zip(rw, hw) if a != b]
            cands.append((len(rw), s, i, test.eng[i], r, h, diff, round(c.sentence_score(h, [r]).score, 1)))
print(exact, "outputs exactly right (words + tone) across 3 seeds")
print(len(cands), "outputs with every word right but tone wrong\n")
for n, s, i, e, r, h, d, sc in sorted(cands)[:10]:
    print(f"seed{s} #{i} | {e}\n  REF: {r}\n  HYP: {h}\n  tone errors: {d} | chrF++: {sc}\n")

# Fallback: short real references with a contested a/à, for a clearly-labelled illustration
print("---- fallback candidates (real references, 3-7 words, containing a / à) ----")
for i, (e, r) in enumerate(zip(test.eng, test.lmp)):
    rw = words(r)
    if 3 <= len(rw) <= 7 and any(w in ("a", "à") for w in rw):
        print(f"#{i} | {e}\n  REF: {r}")

16 outputs exactly right (words + tone) across 3 seeds
0 outputs with every word right but tone wrong

---- fallback candidates (real references, 3-7 words, containing a / à) ----
#110 | There they preached the Good News.
  REF: a ce fè'shi Nsuŋ Mbòŋmboŋ àfo.
#744 | Nfor was here today.
  REF: Nfò à ba ba mbena ntìni.
#746 | The doctor has forbidden him (to eat) pepper.
  REF: Doktà à mʉ baasi ye mnsur.
#748 | I went when he had already gone to market.
  REF: Mɛ̀ dù e banì a du ntaa.
#757 | Where have you kept his books?
  REF: Wèe à lɛ̀ mŋwà' bvii ba fɛ?
#759 | You spoke too much.
  REF: Wɛ̀ làa a càase.
#762 | Whose basket is this?
  REF: Ŋkàa cena a zhì nda?
#763 | Do I always see you?
  REF: Wà' mɛ ke yɛ wɛ̀ cɛ'cɛ' a?
#770 | I have made this chair like this.
  REF: Mɛ̀ kupshi nta' ca à njɛ̀ɛ enɛ̂.
#779 | That road is extremely long.
  REF: Nduunjì anà zhi goni a sap sê.
#787 | I will come before you go.
  REF: M be vʉ̀ wɛɛ à ka' dù.
#791 | Nfuh has formed segment on the pole.
  REF

In [32]:
# %%
# Build the native-speaker evaluation workbook (CPU only, < 1 min).
# Run after: Config -> Drive -> Utilities -> Step 0 -> Step 1a   (config must point at limbum_tone_unicode)
# Compares Direct with Toneless + source-aware ByT5 (the comparison described in the paper),
# blind and in random A/B order, on 100 test sentences that contain contested words and where
# the two systems' outputs differ.
import os, random
import pandas as pd
from openpyxl.styles import Alignment

N_ITEMS, MAX_WORDS, SEED = 100, 30, 0
direct = load_lines(os.path.join(OUT, "runs", "direct_seed1", "test_hyp.txt"))
restore = load_lines(os.path.join(OUT, "restorers", "byt5_src_seed1", "on_stripped_seed1.txt"))
assert len(direct) == len(restore) == len(test)

def has_contested(t): return any(key(w) in lex_amb for w, _ in word_items(t))
cand = [i for i, r in enumerate(test.lmp)
        if has_contested(r) and direct[i] != restore[i] and len(r.split()) <= MAX_WORDS]
rng = random.Random(SEED); rng.shuffle(cand); cand = sorted(cand[:N_ITEMS])
print(f"{len(cand)} items selected")

rows, key_rows = [], []
for n, i in enumerate(cand, 1):
    flip = rng.random() < 0.5
    a, b = (restore[i], direct[i]) if flip else (direct[i], restore[i])
    rows.append({"Item": n, "English": test.eng[i], "Reference Limbum": test.lmp[i],
                 "Output A": a, "Output B": b,
                 "A: number of tone errors": "", "A: does a tone error change the meaning? (yes/no)": "",
                 "B: number of tone errors": "", "B: does a tone error change the meaning? (yes/no)": "",
                 "Which output is better? (A / B / same)": "", "Comments": ""})
    key_rows.append({"Item": n, "test_index": i,
                     "A": "Restore (source-aware ByT5)" if flip else "Direct",
                     "B": "Direct" if flip else "Restore (source-aware ByT5)"})

instructions = pd.DataFrame({"Instructions": [
    "Thank you for helping us evaluate machine translation from English into Limbum.",
    "Each row has an English sentence, a reference Limbum translation, and two machine outputs (A and B).",
    "A and B come from two different systems, shown in random order. Please judge each one independently.",
    "We are mainly interested in TONE MARKS (grave, circumflex, caron), not in other wording differences.",
    "For each output: count the words whose tone marking is wrong, and say whether any tone error changes the meaning.",
    "Then say which output is better overall: A, B, or same.",
    "A different word choice from the reference is fine if it is correct Limbum; only mark it in Comments if it is wrong.",
    "Please use the 'Questions' sheet for a few short questions about Limbum tone.",
]})
questions = pd.DataFrame({"Question": [
    "1. What is the difference between 'a' and 'à' in sentences like 'Woowèe a ko' kùna' (They have gone up there) and 'Ŋgwe à koo yàa mbʉ' (A dog has caught my goat)? Does it depend on the subject (e.g. plural vs. singular)?",
    "2. In 'Nyʉ̀ à m boo rbuu ba nshe' (God made Heaven and Earth), what would 'Nyʉ̀ a m boo rbuu ba nshe' mean, or is it simply wrong?",
    "3. 'Tata' and 'Tatâ' (Lord), and 'Jus' and 'Jûs', both appear in the Bible text. When is the falling tone (circumflex) used? Is it because of the word's position at the end of a phrase?",
    "4. Other pairs that appear with and without tone: laa/làa, gee/gèe, du/dù, yi/yì, enɛ/enɛ̂, mo'/mò'. Are these different words, different grammatical forms, or inconsistent spelling?",
], "Answer": ["", "", "", ""]})

path = os.path.join(OUT, "human_eval", "Limbum_tone_evaluation.xlsx")
os.makedirs(os.path.dirname(path), exist_ok=True)
with pd.ExcelWriter(path, engine="openpyxl") as xw:
    instructions.to_excel(xw, sheet_name="Instructions", index=False)
    pd.DataFrame(rows).to_excel(xw, sheet_name="Evaluation", index=False)
    questions.to_excel(xw, sheet_name="Questions", index=False)
    for ws in xw.book.worksheets:                                   # readable column widths, wrapped text
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = 45 if col[0].column_letter in "BCDE" else 22
            for c in col: c.alignment = Alignment(wrap_text=True, vertical="top")
pd.DataFrame(key_rows).to_csv(os.path.join(OUT, "human_eval", "KEY_do_not_share_v2.csv"), index=False)
print("Share ONLY this file:", path)

100 items selected
Share ONLY this file: /content/drive/MyDrive/Limbum Research/limbum_tone_unicode/human_eval/Limbum_tone_evaluation.xlsx


In [11]:
import pandas as pd
k = pd.read_csv(os.path.join(OUT, "human_eval", "KEY_do_not_share_v2.csv"))
idx = list(k.test_index)
direct = load_lines(os.path.join(OUT, "runs/direct_seed1/test_hyp.txt"))
restore = load_lines(os.path.join(OUT, "restorers/byt5_src_seed1/on_stripped_seed1.txt"))
refs = [test.lmp[i] for i in idx]
for name, h in [("Direct", direct), ("Restore", restore)]:
    hy = [h[i] for i in idx]
    m, per = tone_eval(hy, refs, per_sentence=True)
    errs = [t["m_all"] - t["c_all"] for t in per]
    print(f"{name}: automatic tone mismatches = {sum(errs)}, sentences with >=1 = {sum(e > 0 for e in errs)}, "
          f"aligned words = {m['N_all']}, TAcc = {m['TAcc']:.1f}")

Direct: automatic tone mismatches = 62, sentences with >=1 = 47, aligned words = 1144, TAcc = 94.6
Restore: automatic tone mismatches = 72, sentences with >=1 = 56, aligned words = 1154, TAcc = 93.8
